In [ ]:
# ============================================================
# EXPERIMENT 6A : NAIVE BAYES CLASSIFICATION FROM SCRATCH
# Student Admission Prediction
# ============================================================
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Step 1: Create Dataset
# ------------------------------------------------------------
data = {
    "Study_Hours": [
        "Low", "Low", "Medium", "Medium",
        "High", "High", "High", "Medium",
        "Low", "High"
    ],
    "Attendance": [
        "Low", "High", "Medium", "High",
        "Medium", "High", "High", "Medium",
        "Low", "Medium"
    ],
    "Previous_Marks": [
        "Low", "Medium", "Medium", "High",
        "High", "High", "Medium", "High",
        "Low", "High"
    ],
    "Admission": [
        "No", "No", "No", "Yes",
        "Yes", "Yes", "Yes", "Yes",
        "No", "Yes"
    ]
}

df = pd.DataFrame(data)

print("Student Admission Dataset:")
print(df)

# ------------------------------------------------------------
# Step 2: Separate Features and Target
# ------------------------------------------------------------
features = [
    "Study_Hours",
    "Attendance",
    "Previous_Marks"
]

target = "Admission"

# ============================================================
# Step 3: Calculate Prior Probability
# ============================================================
classes = df[target].unique()
total_samples = len(df)

priors = {}

for cls in classes:
    count = len(df[df[target] == cls])
    priors[cls] = count / total_samples

print("\nPrior Probabilities:")
print("----------------------------------")

for cls, probability in priors.items():
    print(cls, ":", round(probability, 4))

# ============================================================
# Step 4: Conditional Probability
# Using Laplace Smoothing
# ============================================================
def conditional_probability(feature, value, cls):

    class_data = df[df[target] == cls]

    count_feature_value = len(
        class_data[class_data[feature] == value]
    )

    class_count = len(class_data)

    possible_values = df[feature].unique()

    # Laplace smoothing
    # NOTE: PDF shows "class_count + possible_values",
    # which is a typo; must be class_count + len(possible_values)
    probability = (
        count_feature_value + 1
    ) / (
        class_count + len(possible_values)
    )

    return probability

# ============================================================
# Step 5: Display Conditional Probabilities
# ============================================================
print("\nConditional Probabilities:")
print("=============================")

for cls in classes:
    print("\nClass:", cls)

    for feature in features:
        values = df[feature].unique()

        for value in values:
            probability = conditional_probability(feature, value, cls)

            print(
                f"P({feature}={value} | "
                f"Admission={cls}) = "
                f"{probability:.4f}"
            )

# ============================================================
# Step 6: Naive Bayes Prediction Function
# ============================================================
def predict_naive_bayes(sample):

    probabilities = {}

    for cls in classes:
        probability = priors[cls]

        for feature in features:
            value = sample[feature]

            probability *= conditional_probability(
                feature, value, cls
            )

        probabilities[cls] = probability

    prediction = max(probabilities, key=probabilities.get)

    return prediction, probabilities

# ============================================================
# Step 7: Predict New Student
# ============================================================
new_student = {
    "Study_Hours": "High",
    "Attendance": "High",
    "Previous_Marks": "High"
}

prediction, probabilities = predict_naive_bayes(new_student)

# ============================================================
# Steps 8-10: Display Prediction and Normalized Probabilities
# NOTE: These steps are not fully shown in the PDF;
# reconstructed from the printed output.
# ============================================================
print("\n================================")
print("NEW STUDENT")
print("================================")
print("Study Hours :", new_student["Study_Hours"])
print("Attendance :", new_student["Attendance"])
print("Previous Marks :", new_student["Previous_Marks"])

print("\nClass Probabilities:")

for cls, probability in probabilities.items():
    print(cls, ":", round(probability, 6))

print("\n=================================================")
print("NAIVE BAYES PREDICTION")
print("=================================================")
print("Predicted Admission :", prediction)

total_prob = sum(probabilities.values())

print("\nNormalized Probabilities:")

for cls, probability in probabilities.items():
    print(cls, ":", round(probability / total_prob, 4))

# ============================================================
# Step 11: Multiple Predictions
# ============================================================
test_students = pd.DataFrame({
    "Study_Hours": ["High", "Low", "Medium", "High"],
    "Attendance": ["High", "Low", "Medium", "Medium"],
    "Previous_Marks": ["High", "Low", "Medium", "High"]
})

predictions = []

for _, student in test_students.iterrows():

    sample = {
        "Study_Hours": student["Study_Hours"],
        "Attendance": student["Attendance"],
        "Previous_Marks": student["Previous_Marks"]
    }

    prediction, _ = predict_naive_bayes(sample)
    predictions.append(prediction)

test_students["Predicted_Admission"] = predictions

# ============================================================
# Step 12: Display Results
# ============================================================
print("\n=========================================")
print("MULTIPLE STUDENT PREDICTIONS")
print("=========================================")
print(test_students)

In [ ]:
# ============================================================
# EXPERIMENT 6B : GAUSSIAN NAIVE BAYES USING SCIKIT-LEARN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# Step 1: Load Dataset
# ============================================================
iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
target_names = iris.target_names

# ============================================================
# Step 2: Create DataFrame
# ============================================================
df = pd.DataFrame(X, columns=feature_names)

df["Target"] = y
df["Species"] = df["Target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

print("Dataset Shape:")
print(df.shape)

print("\nFirst Five Records:")
print(df.head())

# ============================================================
# Step 3: Dataset Information
# ============================================================
print("\nFeature Names:")
print(feature_names)

print("\nTarget Classes:")
print(target_names)

print("\nClass Distribution:")
print(df["Species"].value_counts())

# ============================================================
# Step 4: Separate Features and Target
# ============================================================
X = iris.data
y = iris.target

# ============================================================
# Step 5: Train-Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# ============================================================
# Steps 6-9: Model, Training, Prediction, Evaluation
# NOTE: These steps are not present in the PDF.
# Reconstructed to match the stated aim.
# ============================================================

# Step 6: Create Gaussian Naive Bayes model
gnb = GaussianNB()

# Step 7: Train
gnb.fit(X_train, y_train)

# Step 8: Predict
y_pred = gnb.predict(X_test)

# Step 9: Actual vs Predicted
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})
results["Actual Species"] = results["Actual"].map(
    dict(enumerate(target_names))
)
results["Predicted Species"] = results["Predicted"].map(
    dict(enumerate(target_names))
)

print("\nActual vs Predicted:")
print(results)

# Step 10: Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\n=================================")
print("GAUSSIAN NAIVE BAYES PERFORMANCE")
print("=================================")
print("Accuracy :", round(accuracy, 4))
print("Accuracy (%) :", round(accuracy * 100, 2))

# Step 11: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Step 12: Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=target_names
))

# Step 13: Confusion Matrix Plot
plt.figure(figsize=(6, 5))

plt.imshow(cm, interpolation="nearest")
plt.title("GaussianNB Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(range(len(target_names)), target_names)
plt.yticks(range(len(target_names)), target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()